In [1]:
from google.colab import files
uploaded = files.upload()


Saving result_2026-10-28-07-06-57.xls to result_2026-10-28-07-06-57.xls


In [2]:
import pandas as pd
import numpy as np
import json
import ast

FILENAME = "result_2026-10-28-07-06-57.xls"   # adjust if different

# Try xlrd (this file is old .xls format based on your earlier file)
try:
    df = pd.read_excel(FILENAME, engine="xlrd")
except Exception:
    !pip install openpyxl -q
    df = pd.read_excel(FILENAME, engine="openpyxl")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print()
print("Routers in file:")
print(df["router_id"].tolist())

Shape: (5, 5)
Columns: ['router_id', 'q_table', 'last_state', 'last_action_idx', 'updated_at']

Routers in file:
['e438191a7dbc', 'e238191a7dbc', 'e638191a7dbc', 'e638191a8bdc', 'e238191a8bdc']


In [3]:
# Find the row with the fully-populated optimized table (e638191a8bdc)
target = df[df["router_id"] == "e638191a8bdc"].iloc[0]
q_table = np.array(ast.literal_eval(target["q_table"]))

print("Q-table shape:", q_table.shape)
print("Non-zero rows:", (np.abs(q_table).sum(axis=1) > 0).sum())
print("Total Q-values:", q_table.size)
print()

# Actions corresponding to columns 0..4 (from Chapter 4)
ACTIONS = [0.80, 0.85, 0.90, 0.95, 1.00]

# State decomposition: state = latency_band * 8 + drops * 4 + flow_load_band
LATENCY_LABELS = ["Low (<50ms)", "Med (50-200ms)", "High (>200ms)"]
DROPS_LABELS   = ["No drops", "Drops"]
LOAD_LABELS    = ["Idle (<0.3)", "Light (0.3-0.6)", "Moderate (0.6-0.8)", "Heavy (>0.8)"]

rows = []
for s in range(24):
    lat_idx  = s // 8
    drop_idx = (s % 8) // 4
    load_idx = s % 4
    q_row = q_table[s]
    if np.abs(q_row).sum() == 0:
        best_action = None
        best_q = None
        status = "UNVISITED"
    else:
        best_action_idx = int(np.argmax(q_row))
        best_action = ACTIONS[best_action_idx]
        best_q = float(q_row[best_action_idx])
        status = "VISITED"
    rows.append({
        "State": s,
        "Latency band": LATENCY_LABELS[lat_idx],
        "Drops": DROPS_LABELS[drop_idx],
        "Load band": LOAD_LABELS[load_idx],
        "Best action": best_action,
        "Best Q-value": round(best_q, 2) if best_q is not None else None,
        "Status": status,
    })

policy_df = pd.DataFrame(rows)
print(policy_df.to_string(index=False))

Q-table shape: (24, 5)
Non-zero rows: 12
Total Q-values: 120

 State   Latency band    Drops          Load band  Best action  Best Q-value    Status
     0    Low (<50ms) No drops        Idle (<0.3)         1.00          0.00   VISITED
     1    Low (<50ms) No drops    Light (0.3-0.6)         0.80        -25.46   VISITED
     2    Low (<50ms) No drops Moderate (0.6-0.8)         0.95         -3.52   VISITED
     3    Low (<50ms) No drops       Heavy (>0.8)         1.00          0.00   VISITED
     4    Low (<50ms)    Drops        Idle (<0.3)          NaN           NaN UNVISITED
     5    Low (<50ms)    Drops    Light (0.3-0.6)          NaN           NaN UNVISITED
     6    Low (<50ms)    Drops Moderate (0.6-0.8)          NaN           NaN UNVISITED
     7    Low (<50ms)    Drops       Heavy (>0.8)          NaN           NaN UNVISITED
     8 Med (50-200ms) No drops        Idle (<0.3)         0.80        -49.40   VISITED
     9 Med (50-200ms) No drops    Light (0.3-0.6)         0.95      